# MP-DocVQA Baseline
## ColQwen2 页面检索 + Qwen2-VL VQA

**Pipeline**
```
用户 Query
   ↓
[ColQwen2-2B]  页面级检索（MaxSim）→ Top-K 页面
   ↓
[Qwen2-VL-7B]  多图 VQA → 答案
```

**运行环境**：AutoDL A100 / 4090，或 Google Colab A100

---
按顺序执行每个 Cell，遇到问题可以单独重跑对应步骤。

## Cell 1 · 安装依赖

In [1]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q colpali-engine datasets qwen-vl-utils pillow tqdm numpy editdistance accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 56.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

In [ ]:
import os

print("✅ HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))

## Cell 3 · 导入库 & 全局配置

> 📌 所有超参数都在这里修改，不需要动后面的代码。

In [ ]:
import os, json, torch, numpy as np, logging
from pathlib import Path
from tqdm.notebook import tqdm   # notebook 友好的进度条
from PIL import Image
from typing import List, Optional
from dataclasses import dataclass, field

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

@dataclass
class Config:
    # ── 模型 ──────────────────────────────────────────────
    retriever_model: str = "vidore/colqwen2-v1.0"
    # 备选检索器（取消注释即可切换）:
    # retriever_model: str = "vidore/colpali-v1.3"       # ColPali（PaliGemma-3B）
    # retriever_model: str = "vidore/colsmol-v0.1"       # 最小，256M

    vlm_model: str = "Qwen/Qwen2-VL-7B-Instruct"
    # 备选 VLM（显存不足时换这个）:
    # vlm_model: str = "Qwen/Qwen2-VL-2B-Instruct"
    # vlm_model: str = "Qwen/Qwen2.5-VL-7B-Instruct"   # 更强

    # ── 数据 ──────────────────────────────────────────────
    dataset_name: str = "nielsr/mp-docvqa"
    split: str        = "validation"
    max_samples: int  = 50          # 快速实验先跑 50，确认 pipeline 后改 -1 跑全量

    # ── 检索 ──────────────────────────────────────────────
    top_k: List[int]         = field(default_factory=lambda: [1, 3, 5])
    retrieval_batch_size: int = 4   # OOM 时调小到 2 或 1

    # ── VQA ───────────────────────────────────────────────
    max_new_tokens: int = 64

    # ── 路径 ──────────────────────────────────────────────
    cache_dir: str   = "./cache"
    results_dir: str = "./results"

    # ── 设备 ──────────────────────────────────────────────
    device: str           = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: torch.dtype    = torch.bfloat16   # A100/4090 用 bfloat16

CFG = Config()
os.makedirs(CFG.cache_dir, exist_ok=True)
os.makedirs(CFG.results_dir, exist_ok=True)

print(f"✅ 设备: {CFG.device}  |  dtype: {CFG.dtype}")
print(f"   检索模型: {CFG.retriever_model}")
print(f"   VQA 模型: {CFG.vlm_model}")
print(f"   数据集:   {CFG.dataset_name} [{CFG.split}], max_samples={CFG.max_samples}")

## Cell 4 · 检查 GPU

In [ ]:
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_gb = props.total_memory / 1e9
        print(f"GPU {i}: {props.name}  |  显存: {total_gb:.1f} GB")
else:
    print("⚠️  未检测到 GPU，将使用 CPU（速度很慢）")

## Cell 5 · 加载数据集

先打印第一条样本的字段名，确认数据结构。  
MP-DocVQA 在 HF 上有几个版本，字段名略有不同，这里会自动检测。

In [ ]:
from datasets import load_dataset

print(f"Loading {CFG.dataset_name} [{CFG.split}]...")
ds = load_dataset(CFG.dataset_name, split=CFG.split, trust_remote_code=True)

if CFG.max_samples > 0:
    ds = ds.select(range(min(CFG.max_samples, len(ds))))

print(f"\n✅ 加载完成，共 {len(ds)} 条样本")
print(f"字段列表: {ds.column_names}")

In [ ]:
# 检查第一条样本，确认字段内容
sample0 = ds[0]
print("=== 第 0 条样本 ===")
for k, v in sample0.items():
    if isinstance(v, Image.Image):
        print(f"  {k}: PIL.Image  size={v.size}")
    elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], Image.Image):
        print(f"  {k}: List[PIL.Image]  len={len(v)}  size[0]={v[0].size}")
    elif isinstance(v, list):
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {repr(v)[:120]}")

In [ ]:
# 根据上面的输出，确认页面图像字段名
# 如果是 "images"（多张），直接用；如果是 "image"（单张），需要特殊处理

def get_page_images(sample) -> List[Image.Image]:
    """从样本中提取所有页面图像，兼容不同版本字段名。"""
    if "images" in sample and sample["images"] is not None:
        imgs = sample["images"]
    elif "image" in sample and sample["image"] is not None:
        imgs = [sample["image"]]
    else:
        raise KeyError(f"找不到图像字段，现有字段: {list(sample.keys())}")

    if not isinstance(imgs, list):
        imgs = [imgs]

    return [img if isinstance(img, Image.Image) else Image.fromarray(img) for img in imgs]

# 测试
test_imgs = get_page_images(ds[0])
print(f"✅ get_page_images 正常，第 0 条文档共 {len(test_imgs)} 页")
print(f"   第 1 页尺寸: {test_imgs[0].size}")

# 可视化第一页
display(test_imgs[0])

## Cell 6 · 加载检索模型（ColQwen2）

第一次运行会从 HuggingFace 下载模型权重（约 5GB），后续会命中缓存。

In [ ]:
from colpali_engine.models import ColQwen2, ColQwen2Processor
# 如果用 ColPali，改成：
# from colpali_engine.models import ColPali, ColPaliProcessor

print(f"Loading {CFG.retriever_model} ...")
retriever_model = ColQwen2.from_pretrained(
    CFG.retriever_model,
    torch_dtype=CFG.dtype,
    device_map=CFG.device,
).eval()
retriever_processor = ColQwen2Processor.from_pretrained(CFG.retriever_model)

print(f"✅ 检索模型加载完成")
print(f"   参数量: {sum(p.numel() for p in retriever_model.parameters()) / 1e9:.2f}B")

## Cell 7 · 检索工具函数

In [ ]:
@torch.no_grad()
def embed_pages(images: List[Image.Image]) -> torch.Tensor:
    """
    对页面图像列表批量编码。
    返回 shape: (N_pages, num_patches, embed_dim)
    """
    all_embs = []
    bs = CFG.retrieval_batch_size
    for i in range(0, len(images), bs):
        batch = images[i : i + bs]
        inputs = retriever_processor.process_images(batch).to(CFG.device)
        embs = retriever_model(**inputs)       # (bs, patches, dim)
        all_embs.append(embs.cpu().float())
    return torch.cat(all_embs, dim=0)         # (N, patches, dim)


@torch.no_grad()
def embed_query(question: str) -> torch.Tensor:
    """
    对单个 query 编码。
    返回 shape: (query_len, embed_dim)
    """
    inputs = retriever_processor.process_queries([question]).to(CFG.device)
    emb = retriever_model(**inputs)            # (1, q_len, dim)
    return emb[0].cpu().float()               # (q_len, dim)


def maxsim_scores(query_emb: torch.Tensor, page_embs: torch.Tensor) -> torch.Tensor:
    """
    ColBERT-style MaxSim：
      score(q, d) = Σ_t  max_p  cos_sim(q_t, d_p)
    输入:
      query_emb : (q_len, dim)
      page_embs : (N, patches, dim)
    输出: (N,)
    """
    q = query_emb.unsqueeze(0)                         # (1, q_len, dim)
    # einsum → (q_len, N, patches)
    sim = torch.einsum("qid,njd->inj", q, page_embs)  # (q_len, N, patches)
    max_sim = sim.max(dim=-1).values                   # (q_len, N)
    scores  = max_sim.sum(dim=0)                       # (N,)
    return scores


def retrieve_top_k(question: str, page_images: List[Image.Image], k: int):
    """
    给定 question 和文档所有页面，返回 Top-K 页面索引（相关性降序）。
    同时返回 query_emb 和 page_embs，方便后续复用，避免重复计算。
    """
    page_embs = embed_pages(page_images)               # (N, patches, dim)
    query_emb = embed_query(question)                  # (q_len, dim)
    scores    = maxsim_scores(query_emb, page_embs)    # (N,)
    top_idx   = scores.argsort(descending=True)[:k].tolist()
    return top_idx, query_emb, page_embs


print("✅ 检索工具函数定义完成")

## Cell 8 · 检索 Smoke Test（用第 0 条样本验证）

In [ ]:
sample = ds[0]
question = sample["question"]
page_images = get_page_images(sample)
gt_page_idx = sample.get("answer_page_idx", None)

print(f"问题: {question}")
print(f"答案: {sample.get('answers', sample.get('answer', '?'))}")
print(f"文档页数: {len(page_images)}")
print(f"正确答案所在页（answer_page_idx）: {gt_page_idx}")
print()

# 跑检索
top_idx, q_emb, p_embs = retrieve_top_k(question, page_images, k=3)
print(f"Top-3 检索结果（页面索引）: {top_idx}")
if gt_page_idx is not None:
    print(f"Hit@1: {gt_page_idx in top_idx[:1]}  |  Hit@3: {gt_page_idx in top_idx}")

# 可视化 Top-1 检索页面
print("\nTop-1 检索到的页面：")
display(page_images[top_idx[0]])

if gt_page_idx is not None and gt_page_idx != top_idx[0]:
    print(f"\n（GT 页面 {gt_page_idx}）：")
    display(page_images[gt_page_idx])

## Cell 9 · 只评测检索（Hit@K）

**建议先跑这个**，比完整 pipeline 快 ~10x，用来快速确认检索质量。  
A100 跑 200 条约 5 分钟。

In [ ]:
hits = {k: [] for k in CFG.top_k}

for sample in tqdm(ds, desc="Retrieval eval"):
    gt_page_idx = sample.get("answer_page_idx", None)
    if gt_page_idx is None:
        continue

    question   = sample["question"]
    page_imgs  = get_page_images(sample)

    for k in CFG.top_k:
        top_idx, _, _ = retrieve_top_k(question, page_imgs, k=k)
        hits[k].append(float(gt_page_idx in top_idx))

print("\n=== 检索结果 Hit@K ===")
for k in CFG.top_k:
    print(f"  Hit@{k}: {np.mean(hits[k]):.4f}  ({int(sum(hits[k]))}/{len(hits[k])})")

## Cell 10 · 加载 VQA 模型（Qwen2-VL）

下载约 15GB（7B 版），首次运行较慢。

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print(f"Loading {CFG.vlm_model} ...")
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    CFG.vlm_model,
    torch_dtype=CFG.dtype,
    device_map="auto",
).eval()
vlm_processor = AutoProcessor.from_pretrained(CFG.vlm_model)

print(f"✅ VLM 加载完成")
print(f"   参数量: {sum(p.numel() for p in vlm.parameters()) / 1e9:.2f}B")

# 打印当前显存占用
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        used  = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"   GPU {i} 显存占用: {used:.1f} / {total:.1f} GB")

## Cell 11 · VQA 工具函数

In [ ]:
@torch.no_grad()
def vqa_answer(question: str, images: List[Image.Image]) -> str:
    """
    给定问题和检索到的页面图像列表，生成答案。
    图像顺序：最相关的在前（对应 Top-1, Top-2, ...）。
    """
    image_content = [{"type": "image", "image": img} for img in images]
    messages = [{
        "role": "user",
        "content": image_content + [{"type": "text", "text": question}],
    }]

    text_prompt   = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
    ).to(CFG.device)

    output_ids = vlm.generate(**inputs, max_new_tokens=CFG.max_new_tokens)
    generated  = output_ids[:, inputs["input_ids"].shape[1]:]
    answer = vlm_processor.batch_decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0].strip()
    return answer


print("✅ VQA 工具函数定义完成")

## Cell 12 · VQA Smoke Test（用第 0 条样本验证）

In [ ]:
sample    = ds[0]
question  = sample["question"]
page_imgs = get_page_images(sample)
answers   = sample.get("answers", [sample.get("answer", "")])
if isinstance(answers, str):
    answers = [answers]

# 用 Top-1 页面做 VQA
top_idx, _, _ = retrieve_top_k(question, page_imgs, k=1)
pred = vqa_answer(question, [page_imgs[top_idx[0]]])

print(f"问题:    {question}")
print(f"预测:    {pred}")
print(f"GT 答案: {answers}")

## Cell 13 · ANLS 评测指标

In [ ]:
try:
    import editdistance
except ImportError:
    raise ImportError("请先运行 Cell 1 安装依赖")

def anls_score(prediction: str, gold_answers: List[str], threshold: float = 0.5) -> float:
    """
    ANLS (Average Normalized Levenshtein Similarity)
    MP-DocVQA 官方指标，取预测与所有 GT 答案中最高的 NLS。
    """
    def nls(pred: str, gold: str) -> float:
        pred, gold = pred.lower().strip(), gold.lower().strip()
        if not pred and not gold:
            return 1.0
        dist   = editdistance.eval(pred, gold)
        max_l  = max(len(pred), len(gold))
        nl     = dist / max_l
        return 1.0 - nl if nl < (1 - threshold) else 0.0

    return max(nls(prediction, g) for g in gold_answers)


# 快速验证
assert anls_score("Paris", ["Paris"])    == 1.0
assert anls_score("Paris", ["paris"])    == 1.0
assert anls_score("Lyon",  ["Paris"])    == 0.0
print("✅ ANLS 函数验证通过")

## Cell 14 · 完整 Pipeline 评测

同时评测：
- **ANLS**（VQA 质量）
- **Hit@K**（页面检索召回率）

每 10 条打印一次中间结果，方便随时中断查看。

In [ ]:
results      = {k: {"anls": [], "hit": []} for k in CFG.top_k}
all_preds    = []

for idx, sample in enumerate(tqdm(ds, desc="Full pipeline")):
    question    = sample["question"]
    answers     = sample.get("answers", [sample.get("answer", "")])
    if isinstance(answers, str):
        answers = [answers]
    gt_page_idx = sample.get("answer_page_idx", None)
    page_imgs   = get_page_images(sample)

    # 一次性算好 embedding，所有 K 值复用
    page_embs = embed_pages(page_imgs)
    query_emb = embed_query(question)
    scores    = maxsim_scores(query_emb, page_embs)

    for k in CFG.top_k:
        top_idx = scores.argsort(descending=True)[:k].tolist()
        retrieved_imgs = [page_imgs[i] for i in top_idx]

        pred  = vqa_answer(question, retrieved_imgs)
        anls  = anls_score(pred, answers)
        results[k]["anls"].append(anls)

        if gt_page_idx is not None:
            results[k]["hit"].append(float(gt_page_idx in top_idx))

    # 记录 Top-1 预测
    top1_idx = scores.argsort(descending=True)[:1].tolist()
    all_preds.append({
        "idx": idx, "question": question,
        "answers": answers, "gt_page_idx": gt_page_idx,
        "top1_page": top1_idx[0],
        "prediction": vqa_answer(question, [page_imgs[top1_idx[0]]]),
    })

    # 每 10 条打印中间结果
    if (idx + 1) % 10 == 0:
        print(f"\n[{idx+1}/{len(ds)}]")
        for k in CFG.top_k:
            a = np.mean(results[k]["anls"])
            h = np.mean(results[k]["hit"]) if results[k]["hit"] else float("nan")
            print(f"  Top-{k}: ANLS={a:.4f}  Hit@{k}={h:.4f}")

## Cell 15 · 汇总结果 & 保存

In [ ]:
print("\n" + "="*55)
print("FINAL RESULTS")
print("="*55)

summary = {}
for k in CFG.top_k:
    avg_anls = float(np.mean(results[k]["anls"]))
    avg_hit  = float(np.mean(results[k]["hit"])) if results[k]["hit"] else float("nan")
    summary[f"top_{k}"] = {"anls": avg_anls, f"hit@{k}": avg_hit}
    print(f"  Top-{k:2d}  |  ANLS: {avg_anls:.4f}  |  Hit@{k}: {avg_hit:.4f}")

print("="*55)

# 保存
out_path = Path(CFG.results_dir) / "results_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

pred_path = Path(CFG.results_dir) / "predictions.json"
with open(pred_path, "w") as f:
    json.dump(all_preds, f, indent=2, ensure_ascii=False)

print(f"\n✅ 结果已保存到 {CFG.results_dir}/")

## Cell 16 · 错误案例分析

查看 ANLS < 0.3 的样本，帮助定位问题。

In [ ]:
# 只分析 Top-1 结果
k = 1
anls_list = results[k]["anls"]

# 找出分数较低的样本
bad_idx = [i for i, s in enumerate(anls_list) if s < 0.3]
print(f"ANLS < 0.3 的样本数: {len(bad_idx)} / {len(anls_list)}")

# 随机展示 3 条
import random
show = random.sample(bad_idx, min(3, len(bad_idx)))

for i in show:
    p = all_preds[i]
    print(f"\n{'─'*50}")
    print(f"问题:    {p['question']}")
    print(f"GT 答案: {p['answers']}")
    print(f"预测:    {p['prediction']}")
    print(f"GT 页:   {p['gt_page_idx']}  |  检索到页: {p['top1_page']}")
    print(f"ANLS:    {anls_list[i]:.4f}")

    # 显示检索到的页面和 GT 页面
    sample    = ds[i]
    page_imgs = get_page_images(sample)
    print("检索到的 Top-1 页面：")
    display(page_imgs[p['top1_page']])
    if p['gt_page_idx'] is not None and p['gt_page_idx'] != p['top1_page']:
        print(f"GT 页面（第 {p['gt_page_idx']} 页）：")
        display(page_imgs[p['gt_page_idx']])

## Cell 17 · 消融：不同 Top-K 对 ANLS 和 Hit@K 的影响

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['font.family'] = 'DejaVu Sans'  # 防止中文乱码

    k_vals  = CFG.top_k
    anls_vals = [float(np.mean(results[k]["anls"])) for k in k_vals]
    hit_vals  = [float(np.mean(results[k]["hit"])) if results[k]["hit"] else 0.0 for k in k_vals]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.bar([str(k) for k in k_vals], anls_vals, color="#4C72B0")
    ax1.set_title("ANLS vs Top-K")
    ax1.set_xlabel("Top-K pages fed to VLM")
    ax1.set_ylabel("ANLS")
    ax1.set_ylim(0, 1)
    for i, v in enumerate(anls_vals):
        ax1.text(i, v + 0.01, f"{v:.3f}", ha="center")

    ax2.bar([str(k) for k in k_vals], hit_vals, color="#DD8452")
    ax2.set_title("Hit@K (Page Recall)")
    ax2.set_xlabel("K")
    ax2.set_ylabel("Hit@K")
    ax2.set_ylim(0, 1)
    for i, v in enumerate(hit_vals):
        ax2.text(i, v + 0.01, f"{v:.3f}", ha="center")

    plt.tight_layout()
    plt.savefig(f"{CFG.results_dir}/ablation_topk.png", dpi=150)
    plt.show()
    print("✅ 图已保存")
except ImportError:
    print("matplotlib 未安装，跳过可视化")